# Prompt tuning learns vectors at the input. Prefix tuning learns attention key/value prefixes for transformer layers.

# Domain Adaptation through Prefix Tuning

In attention, a model uses queries, keys, and values.

Prefix tuning supplies additional learned keys and values
to the attention mechanism at each transformer layer.

Conceptually:

Attention(
    Queries from input,
    [Prefix keys + Input keys],
    [Prefix values + Input values]
)

During training:
- All original model parameters remain frozen.
- Only the prefix parameters are updated.

The prefix contains numerical vectors, not readable instructions.

In [25]:
%pip install -q torch "transformers>=4.45,<5" "peft>=0.13,<0.19" pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import hashlib
import math
import random

import pandas as pd
import torch

from IPython.display import display
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

from peft import (
    PrefixTuningConfig,
    TaskType,
    get_peft_model,
)

# Choose "cpu", "cuda", or "auto".
DEVICE_CHOICE = "cpu"

if DEVICE_CHOICE == "auto":
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
elif DEVICE_CHOICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError(
            'CUDA unavailable. Set DEVICE_CHOICE = "cpu".'
        )
    device = torch.device("cuda")
elif DEVICE_CHOICE == "cpu":
    device = torch.device("cpu")
else:
    raise ValueError('Choose "cpu", "cuda", or "auto".')

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [27]:
MODEL_NAME = "./workshop_models/distilgpt2"

## Create 10 virtual prefix positions for attention in each transformer layer.

NUM_VIRTUAL_TOKENS = 10

# Learn the prefix key/value vectors directly.
# False means we do not use an additional projection network.
PREFIX_PROJECTION = False

MAX_LENGTH = 96
BATCH_SIZE = 1
TRAINING_STEPS = 30
LEARNING_RATE = 1e-2

print("Prefix positions:", NUM_VIRTUAL_TOKENS)
print("Training steps:", TRAINING_STEPS)

Prefix positions: 10
Training steps: 30


In [28]:
# Training and evaluation data
# These are the same teaching examples which i had used previously.

In [29]:
train_texts = [
    "Predictive maintenance uses equipment condition data to plan "
    "maintenance before a failure occurs.",

    "Preventive maintenance follows a planned schedule based on "
    "time or equipment usage.",

    "Vibration monitoring helps engineers identify changes in "
    "the condition of rotating machinery.",

    "Bearing faults can produce characteristic vibration patterns. "
    "Engineers examine frequency information and operating conditions.",

    "An increase in bearing temperature may indicate lubrication "
    "problems, increased load, or other developing issues.",

    "Root mean square vibration summarizes the magnitude of a "
    "vibration signal over a measurement interval.",

    "A condition monitoring baseline describes normal equipment "
    "behavior under known operating conditions.",

    "Sensor trends should be compared with operating speed and load "
    "because these factors can influence measurements.",

    "A single abnormal reading does not confirm a machine fault. "
    "Additional measurements help support a diagnosis.",

    "Lubrication reduces friction and wear between moving surfaces. "
    "Incorrect lubrication can contribute to bearing damage.",

    "Shaft misalignment can increase vibration and mechanical stress "
    "in rotating equipment.",

    "Maintenance decisions should combine sensor measurements, "
    "inspection findings, and equipment history.",
]

eval_texts = [
    "Condition-based maintenance uses observations of machine health "
    "to decide when maintenance is needed.",

    "A technician investigating elevated vibration should consider "
    "changes in load, speed, alignment, and bearing condition.",

    "Rising bearing temperature is a symptom that requires further "
    "investigation rather than an immediate fault diagnosis.",

    "Comparing measurements with a normal operating baseline helps "
    "identify changes in equipment condition.",
]

assert set(train_texts).isdisjoint(eval_texts)

print("Training texts:", len(train_texts))
print("Evaluation texts:", len(eval_texts))

Training texts: 12
Evaluation texts: 4


In [30]:
# Load a fresh model and prepare batches

In [31]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# tokenizer.pad_token = tokenizer.eos_token

# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     attn_implementation="eager",
# )

# base_model.config.pad_token_id = tokenizer.pad_token_id
# base_model.config.use_cache = True

# base_model = base_model.to(device)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    attn_implementation="eager",
).to(device)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = True


def collate_texts(texts):
    batch = tokenizer(
        [text + tokenizer.eos_token for text in texts],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100

    batch["labels"] = labels
    return batch


train_loader = DataLoader(
    train_texts,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_texts,
)

eval_loader = DataLoader(
    eval_texts,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_texts,
)

print("Model and data loaders ready.")

Model and data loaders ready.


In [32]:
# Evaluation and completion functions
## Unlike soft input prompts, key/value prefixes do not add output-logit positions. We score the real input tokens using the usual next-token shift.

In [33]:
@torch.no_grad()
def evaluate_loss(current_model, loader):
    current_model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        output = current_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

        prediction_logits = output.logits[:, :-1, :].contiguous()
        targets = batch["labels"][:, 1:].contiguous()

        loss_sum = torch.nn.functional.cross_entropy(
            prediction_logits.reshape(-1, prediction_logits.shape[-1]),
            targets.reshape(-1),
            ignore_index=-100,
            reduction="sum",
        )

        total_loss += loss_sum.item()
        total_tokens += (targets != -100).sum().item()

    return total_loss / total_tokens


@torch.no_grad()
def complete_text(current_model, prompt):
    current_model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)

    output_ids = current_model.generate(
        **inputs,
        max_new_tokens=45,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    prompt_length = inputs["input_ids"].shape[1]

    return tokenizer.decode(
        output_ids[0, prompt_length:],
        skip_special_tokens=True,
    ).strip()


test_prompts = [
    "Predictive maintenance is",
    "When bearing temperature increases,",
    "Vibration monitoring helps engineers",
]

In [34]:
# Record the original model baseline

In [35]:
original_loss = evaluate_loss(base_model, eval_loader)

original_completions = {
    prompt: complete_text(base_model, prompt)
    for prompt in test_prompts
}

print(f"Original model loss: {original_loss:.4f}")
print(f"Original model perplexity: {math.exp(original_loss):.2f}")

for prompt, completion in original_completions.items():
    print("\nPROMPT:", prompt)
    print("CONTINUATION:", completion)

Original model loss: 5.4397
Original model perplexity: 230.38

PROMPT: Predictive maintenance is
CONTINUATION: a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is

PROMPT: When bearing temperature increases,
CONTINUATION: the temperature increases, and the temperature decreases.

PROMPT: Vibration monitoring helps engineers
CONTINUATION: to understand the impact of the stress on the brain.


In [36]:
# Attach trainable prefixes

In [37]:
# Configure prefix tuning for next-token prediction.
prefix_config = PrefixTuningConfig(
    # Our base model is a causal language model.
    task_type=TaskType.CAUSAL_LM,
    # Each layer receives 10 additional key/value prefix positions.
    num_virtual_tokens=NUM_VIRTUAL_TOKENS,
    # Directly train the prefix vectors without a projection network.
    prefix_projection=PREFIX_PROJECTION,
)

# Wrap the base model with PEFT.
# PEFT creates the trainable prefix parameters and freezes the base-model weights.
# During forward passes, PEFT supplies the prefix as attention key/value states.
model = get_peft_model(base_model, prefix_config)
# Move the base model and prefix parameters to the selected CPU or GPU.
model = model.to(device)
# Display how many parameters are trainable.
model.print_trainable_parameters()
# Verify that every original base-model parameter is frozen.
assert all(
    not parameter.requires_grad
    for parameter in model.base_model.parameters()
)
# Display the names, shapes, and sizes of the trainable prefix parameters.
trainable_table = pd.DataFrame([
    {
        "parameter": name,
        "shape": str(tuple(parameter.shape)),
        "elements": parameter.numel(),
    }
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
])

display(trainable_table)


# Save independent copies of the prefix parameters before training.
# These copies let us verify the actual changes after training.
prefix_before = {
    name: parameter.detach().cpu().clone()
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
}

trainable params: 92,160 || all params: 82,004,736 || trainable%: 0.1124


,parameter,shape,elements
0,prompt_encoder.default.embedding.weight,"(10, 9216)",92160


$
N_{\text{trainable}}=P\times L\times 2\times d
$
where \(P\) is prefix length, \(L\) is the number of layers, and \(d\) is the hidden dimension. The factor 2 represents keys and values.

In [38]:
# Read the number of transformer layers: (In this case, 6 for DistilGPT2.
layers = base_model.config.n_layer
# Read the hidden dimension: 768 for DistilGPT2.
hidden_size = base_model.config.n_embd

# With projection disabled, every prefix position stores:
# one key vector and one value vector for each transformer layer.
# Count = prefix positions × layers × 2 (key and value) × hidden dimension.
expected_count = (
    NUM_VIRTUAL_TOKENS * layers * 2 * hidden_size
)
# Count the parameters that are actually trainable.
actual_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Layers:", layers)
print("Hidden dimension:", hidden_size)
print("Expected trainable parameters:", expected_count)
print("Actual trainable parameters:", actual_count)

# Verify that the actual count matches our calculation.
assert actual_count == expected_count
# For this configuration: 10 × 6 × 2 × 768 = 92,160 parameters.

Layers: 6
Hidden dimension: 768
Expected trainable parameters: 92160
Actual trainable parameters: 92160


Prompt tuning: $(10 \times 768 = 7,680)$

Prefix tuning: $(10 \times 6 \times 2 \times 768 = 92,160)$

In [39]:
### Inspecting the attention prefixes

# Switch to evaluation mode for inspection.
model.eval()
# Inspect the prefix without recording gradients.
with torch.no_grad():
    # PEFT reshapes the learned prefix parameters into attention key/value states.
    attention_prefix = model.get_prompt(batch_size=1)
# just ispecting how many layer prefixes were produced.
print("Number of layer prefixes:", len(attention_prefix))

# Select the prefix for the first transformer layer.
first_layer = attention_prefix[0]

# Extract its key and value tensors.
# The representation can differ between library versions.
if isinstance(first_layer, (tuple, list)):
    keys, values = first_layer[:2]
else:
    # Some versions stack keys and values along the first dimension.
    keys, values = first_layer[0], first_layer[1]
    
# Each tensor has dimensions:
# batch size × attention heads × prefix positions × head dimension.
print("First layer key shape:  ", tuple(keys.shape))
print("First layer value shape:", tuple(values.shape))

# For DistilGPT2 with batch size 1, the expected shape should be (1, 12, 10, 64).
# The 12 attention heads together account for 12 × 64 = 768 dimensions.

Number of layer prefixes: 6
First layer key shape:   (1, 12, 10, 64)
First layer value shape: (1, 12, 10, 64)


In [40]:
# Record the initialized prefix baseline
## Adding randomly initialized prefixes may initially hurt performance. 
# Measuring this separately from training.

In [41]:
def frozen_parameter_fingerprints(current_model):
    return {
        name: hashlib.sha256(
            parameter.detach().cpu().contiguous().numpy().tobytes()
        ).hexdigest()
        for name, parameter in current_model.named_parameters()
        if not parameter.requires_grad
    }


frozen_before = frozen_parameter_fingerprints(model)

initialized_loss = evaluate_loss(model, eval_loader)

initialized_completions = {
    prompt: complete_text(model, prompt)
    for prompt in test_prompts
}

print(f"Initialized prefix loss: {initialized_loss:.4f}")

C:\Users\fedex\anaconda3\envs\ABMLLM\Lib\site-packages\peft\peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


Initialized prefix loss: 10.0874


In [42]:
# Train the prefixes

In [43]:
# Collect only parameters that are allowed to change.
# In this configuration, these belong to the prefix encoder.
trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

# Give only the trainable prefix parameters to the optimizer.
# Original model weights are excluded.
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.0,
)

training_history = []
train_iterator = iter(train_loader)

model.train()

for step in range(1, TRAINING_STEPS + 1):
    try:
        batch = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        batch = next(train_iterator)

    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }
    # Clear gradients from the previous training step.
    optimizer.zero_grad(set_to_none=True)
    # Run the forward pass.
    # PEFT internally supplies the learned prefix as past_key_values.
    # Each layer's real-token queries can attend to the additional prefix keys.
    # The corresponding prefix values contribute to the attention output.
    output = model(**batch)
    
    # Retrieve the next-token prediction loss for the real input text.
    # Prefix positions do not create additional output-logit positions.
    loss = output.loss
    # Backpropagate through the frozen model to calculate prefix gradients.
    # The original model parameters remain frozen.
    loss.backward()
    # Limit gradient magnitude to help stabilize prefix training.
    torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        max_norm=1.0,
    )
    # Actually update the trainable prefix parameters.
    optimizer.step()

    training_history.append({
        "step": step,
        "training_loss": loss.item(),
    })

    if step == 1 or step % 5 == 0 or step == TRAINING_STEPS:
        print(
            f"Step {step:3d}/{TRAINING_STEPS} "
            f"| Training loss: {loss.item():.4f}"
        )

print("\nPrefix tuning complete.")

Step   1/30 | Training loss: 9.9671
Step   5/30 | Training loss: 9.9340
Step  10/30 | Training loss: 6.7192
Step  15/30 | Training loss: 7.5356
Step  20/30 | Training loss: 6.7874
Step  25/30 | Training loss: 5.8718
Step  30/30 | Training loss: 6.1889

Prefix tuning complete.


In [44]:
# Compare evaluation results and completions

In [45]:
trained_loss = evaluate_loss(model, eval_loader)

results_df = pd.DataFrame([
    {
        "stage": "Original model",
        "evaluation_loss": original_loss,
        "perplexity": math.exp(original_loss),
    },
    {
        "stage": "Initialized prefix",
        "evaluation_loss": initialized_loss,
        "perplexity": math.exp(initialized_loss),
    },
    {
        "stage": "Trained prefix",
        "evaluation_loss": trained_loss,
        "perplexity": math.exp(trained_loss),
    },
])

display(results_df)

print(
    "Loss reduction from prefix training:",
    f"{initialized_loss - trained_loss:.4f}",
)

print(
    "Loss reduction versus original model:",
    f"{original_loss - trained_loss:.4f}",
)

trained_completions = {
    prompt: complete_text(model, prompt)
    for prompt in test_prompts
}

for prompt in test_prompts:
    print("\nPROMPT:", prompt)

    print("\nORIGINAL MODEL:")
    print(original_completions[prompt])

    print("\nINITIALIZED PREFIX:")
    print(initialized_completions[prompt])

    print("\nTRAINED PREFIX:")
    print(trained_completions[prompt])

    print("=" * 70)

,stage,evaluation_loss,perplexity
0,Original model,5.439722,230.378014
1,Initialized prefix,10.087388,24037.912099
2,Trained prefix,5.793080,328.021867


Loss reduction from prefix training: 4.2943
Loss reduction versus original model: -0.3534


C:\Users\fedex\anaconda3\envs\ABMLLM\Lib\site-packages\peft\peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")



PROMPT: Predictive maintenance is

ORIGINAL MODEL:
a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is a key component of the software. It is

INITIALIZED PREFIX:
a 3.

TRAINED PREFIX:
a risk.

PROMPT: When bearing temperature increases,

ORIGINAL MODEL:
the temperature increases, and the temperature decreases.

INITIALIZED PREFIX:


TRAINED PREFIX:
is the performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance of performance

PROMPT: Vibration monitoring helps engineers

ORIGINAL MODEL:
to understand the impact of the stress on the brain.

INITIALIZED PREFIX:


TRAINED PREFIX:
and the performance.


In [46]:
### Lower evaluation loss indicates better prediction of these held-out sentences. It does not establish factual correctness or broad domain expertise.

In [47]:
# Verifying which parameters changed

In [48]:
# Recalculate fingerprints of all frozen parameter tensors.
# frozen_before was recorded before training in your earlier notebook cell.

frozen_after = frozen_parameter_fingerprints(model)

# Find any frozen parameter tensors whose values changed.

changed_frozen = [
    name
    for name in frozen_before
    if frozen_before[name] != frozen_after[name]
]

change_rows = []

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
         # Read the current prefix parameter values.
        after = parameter.detach().cpu()
         # Compare them with the saved values from before training.
        difference = after - prefix_before[name]

        change_rows.append({
            "parameter": name,
            "changed": not torch.equal(after, prefix_before[name]),
            "mean_absolute_change": difference.abs().mean().item(),
        })

change_df = pd.DataFrame(change_rows)

display(change_df)
print("Changed frozen tensors:", len(changed_frozen))
# Confirm that at least one trainable prefix tensor changed.
assert change_df["changed"].any()
# Confirm that every frozen parameter tensor stayed identical.
assert len(changed_frozen) == 0

print("\nVerified: prefix parameters changed; base parameters stayed identical.")

,parameter,changed,mean_absolute_change
0,prompt_encoder.default.embedding.weight,True,0.072102


Changed frozen tensors: 0

Verified: prefix parameters changed; base parameters stayed identical.


| Method | What we train | Where it enters the model |
|---|---|---|
| Prompt tuning | Extra input embedding vectors | Before real-token embeddings at the input |
| Prefix tuning | Extra attention key/value vectors | In the attention mechanism of each transformer layer |

# 
## Participant exercises

1. Increase NUM_VIRTUAL_TOKENS from 10 to 20.
   Predict the trainable parameter count before running.

2. Compare 30 and 60 training steps.

3. Try LEARNING_RATE = 1e-3.

4. Why can an initialized prefix perform worse than the original model?

5. Why does prefix tuning use more parameters than input prompt tuning
   in this configuration?

For a fresh experiment:
Change settings in Cell 4, then rerun Cells 6–14.

## Methods studied so far

BitFit:
Train existing bias parameters.

Prompt tuning:
Train soft prompt embeddings at the input.

Prefix tuning:
Train attention key/value prefixes across transformer layers.

## Main takeaway

Prefix tuning changes the attention context available to the model
while keeping the original model parameters frozen.